# Analysis of genome-scale metabolic model (GEM) of *L. mesenteroides* (ATCC 8293)

**Project:** Cooperative Tradeoff Analysis of Consortia for Plant-Based Fermentation 

The *Leuconostoc mesenteroides* model - Koduru2022.xml - was was obtained from BioModels (https://biomodels.org/MODEL2210190012). A second model - iLME620.xml - was retrieved from the supplementary data from its original scientific article (https://pmc.ncbi.nlm.nih.gov/articles/PMC5691038/#Sec2)

The analysis performed to evaluate the model includes:
* **memote score** - final score of 26% (Koduru2022) and 44% (iLME620)
* **biomass**, **mass leaks**, **reactions/genes/metabolites**, **blocked reactions**.

## 1. Setup

In [1]:
# Setup 
import os, sys, cobra, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='memote')

_lic = os.path.expanduser('~/gurobi.lic')
if os.path.exists(_lic):
    os.environ['GRB_LICENSE_FILE'] = _lic
cobra.Configuration().solver = 'gurobi'

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import gem_summary

## 2. Models' Quality Analysis

In [2]:
#quality analysis of Koduru2022
model_lm = cobra.io.read_sbml_model('../models/raw/Koduru2022.xml')
results_lm = gem_summary(model_lm, label='Koduru2022')

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05


http://purl.obolibrary.org/obo/GO_0016052 does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id
/Users/Sara_1/miniconda3/envs/sysbio/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpbqda9_dj.lp
Reading time = 0.00 seconds
: 916 rows, 2189 columns, 9235 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpobhlk8hk.lp
Reading time = 0.00 seconds
: 916 rows, 2189 columns, 9235 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q8000

In [3]:
#quality analysis of iLME620
model_lm1 = cobra.io.read_sbml_model('../models/raw/iLME620.xml')
results_lm1 = gem_summary(model_lm1, label='iLME620')

/Users/Sara_1/miniconda3/envs/sysbio/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpw4q1wmvd.lp
Reading time = 0.00 seconds
: 745 rows, 1711 columns, 7397 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp5h2n33p5.lp
Reading time = 0.00 seconds
: 745 rows, 1711 columns, 7397 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q8000

Neither model could be considered fully valid on initial inspection: iLME620 carries no gene-protein-reaction associations and the original Koduru2022 produces zero growth. To diagnose the cause of the latter, a recommendation was made to run BioISO - a computational tool that evaluates biomass or metabolic network formulation in constraint-based models using a recursive, FBA-based algorithm to localize gaps and errors.

## 3. Root cause diagnosis

### 3.1 BioISO Biomass Diagnosis

BioISO was run on Koduru2022 to identify which biomass components were preventing growth. Of the 18 reactants required for the biomass reaction, 12 returned a negative evaluation:

`CRB_LME_c`, `DNA_LME_c`, `GLIP_LME_c`, `LTA_LME_c`, `PLIP_LME_c`, `PROT_LME_c`, `RNA_LME_c`, `btn_c`, `pdx5p_c`, `peptido_LME_c`, `ribflv_c`, `udcpdp_c`

The remaining 6 (`atp_c`, `coa_c`, `h2o_c`, `nad_c`, `thglu3_c`, `thmmp_c`) passed evaluation.

Note: the BioISO report lists this metabolite as peptido_LME_c, omitting the index suffix present in the model's actual ID (peptido1_LME_c/peptido1-LME_c). This is a naming simplification in the BioISO output, not a distinct metabolite.

### 3.2 Critical Gap Identification

The 12 components flagged by BioISO could be independent gaps or have dependencies between them. To distinguish critical from non-critical gaps, exchange reactions were added for all 18 biomass macromolecules, then removed one at a time to test whether the model could still grow without each one individually.

In [4]:
blocked_compounds = [
    'CRB_LME_c', 'DNA_LME_c', 'GLIP_LME_c', 'LTA_LME_c', 'PLIP_LME_c', 'PROT_LME_c', 'RNA_LME_c', 'peptido_LME_c', 
    "btn_c", "pdx5p_c", "ribflv_c", "udcpdp_c", "atp_c", "coa_c", "h2o_c", "nad_c", "thglu3_c", "thmmp_c",
]

# model with all externally available blocked compounds
model_todas = model_lm.copy()
for met_id in blocked_compounds:
    met = model_todas.metabolites.get_by_id(met_id)
    ex = cobra.Reaction(f'EX_{met_id}')
    ex.lower_bound = -1000 
    ex.upper_bound = 1000
    model_todas.add_reactions([ex])
    ex.add_metabolites({met: -1.0})

sol_todas = model_todas.optimize()
print(f"Growth with all available compounds: {sol_todas.objective_value:.4f}")

# remove one at a time and see the impact on growth
for met_id in blocked_compounds:
    model_test = model_lm.copy() 
    
    # add all exchanges except the one we are testing
    for outro_met_id in blocked_compounds:
        if outro_met_id != met_id:
            met = model_test.metabolites.get_by_id(outro_met_id)
            ex = cobra.Reaction(f'EX_{outro_met_id}')
            ex.lower_bound = -1000
            ex.upper_bound = 1000
            model_test.add_reactions([ex])
            ex.add_metabolites({met: -1.0})
    
    sol = model_test.optimize()
    status = "No growth" if sol.objective_value < 0.01 else "presents growth, therefore it's not critical"
    print(f"  Without {met_id}: growth = {sol.objective_value:.4f}  {status}")


         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpbf8s_n0f.lp
Reading time = 0.00 seconds
: 915 rows, 2188 columns, 9232 nonzeros
Growth with all available compounds: 1.8518
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpwyuni8jp.lp
Reading time = 0.00 seconds
: 915 rows, 2188 columns, 9232 nonzeros
  Without CRB_LME_c: growth = 1.3454  presents growth, therefore it's not critical
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpvcfih351.lp
Reading time = 0.00 seconds
: 915 rows, 2188 columns, 9232 nonzeros
  Without DNA_LME_c: growth = 1.7395  presents growth, therefore it's not critical
         of a constant term in the left-hand side of a constraint.

Read LP format model from fil

In [5]:
with model_lm:
    peptido1 = model_lm.metabolites.get_by_id('peptido1_LME_c')
    ex = cobra.Reaction('EX_peptido1_LME_c')
    ex.lower_bound = -1000
    ex.upper_bound = 0
    ex.add_metabolites({peptido1: -1.0})
    model_lm.add_reactions([ex])

    sol = model_lm.optimize()
    print(f"Growth with peptido supplied externally: {sol.objective_value:.4f} h⁻¹")
    if sol.objective_value > 0.5:
        print("Growth restored - peptido1_LME_c is the sole critical gap")

Growth with peptido supplied externally: 0.5912 h⁻¹
Growth restored - peptido1_LME_c is the sole critical gap


In [6]:
for r in model_lm.reactions:
    met_ids = [m.id for m in r.metabolites]
    if any('peptido' in m.lower() for m in met_ids):
        print(f"\n{r.id}: {r.reaction}")
        print(f"  Genes: {[g.id for g in r.genes]}")


biomass: 0.00011235 CRB_LME_c + 9.362e-05 DNA_LME_c + 2.076e-05 GLIP_LME_c + 1.486e-05 LTA_LME_c + 2.063e-05 PLIP_LME_c + 0.002687 PROT_LME_c + 0.00022797 RNA_LME_c + 13.613 atp_c + 1e-05 btn_c + 0.0002 coa_c + 13.613 h2o_c + 0.002 nad_c + 1e-05 pdx5p_c + 0.11907 peptido_LME_c + 1e-05 ribflv_c + 1e-05 thglu3_c + 1e-05 thmmp_c + 0.0002 udcpdp_c --> 13.613 adp_c + 13.613 h_c + 13.613 pi_c
  Genes: []

PPTGS: uaaGgla_c --> peptido1-LME_c + udcpdp_c
  Genes: ['LEUM_RS05925', 'LEUM_RS03180']

SDADACP: h2o_c + peptido1_LME_c --> ala-D_c + peptido_LME_c
  Genes: ['LEUM_RS09815', 'LEUM_RS01410', 'LEUM_RS01710']


The sensitivity analysis identified `peptido1_LME_c` as the sole critical gap in the Koduru2022 biomass chain: removing it from the set of externally supplied compounds was the only condition that abolished growth entirely. All other 17 blocked biomass components are non-critical as the model grows in their absence.

Inspection of the reactions involving `peptido1_LME_c` revealed the cause: PPTGS produces `peptido1-LME_c` (hyphen) while SDADACP consumes
`peptido1_LME_c` (underscore). The solver treats these as two distinct metabolites, leaving PPTGS with no consumer and SDADACP with no producer. The peptidoglycan biosynthesis chain is broken at this single point, which is sufficient to block the entire biomass reaction.

Supplying `peptido1_LME_c` externally confirmed this: growth was fully restored to 0.59 h⁻¹, consistent with published *L. mesenteroides* growth rates in rich medium.

This is consistent with the BioISO diagnosis in Section 3.1, which flagged the same metabolite (under the name `peptido_LME_c`) as part of the blocked biomass chain - the present analysis identifies the specific cause.

## 4. Model Selection

Two genome-scale models were evaluated for *Leuconostoc mesenteroides*: iLME620 and Koduru2022.

| Criterion | iLME620 | Koduru2022 |
|---|---|---|
| Reactions | 855 | 1094 |
| Metabolites | 744 | 915 |
| Genes (GPRs) | 0 | 687 |
| Growth (default medium) | 0.68 h⁻¹ | 0.0 h⁻¹ |
| Blocked reactions | 303 (35.4%) | 654 (59.8%) |
| Memote score | 44% | 26% 
| Root cause of failure | No GPRs | peptido1_LME_c ID mismatch |

The iLME620 model carries no gene-protein-reaction associations, meaning the model was published without GPRs. As a consequence, gene essentiality analysis, flux coupling to genomic evidence and any gene-level interpretation of simulation results are not possible. For a project that aims to connect metabolic flux predictions to specific enzymatic capacity in the consortium, this is a limitation.

The Koduru2022 model has zero growth in its original version due to a single metabolite ID inconsistency: `peptido1-LME_c` (hyphen, produced by PPTGS) and `peptido1_LME_c` (underscore, consumed by SDADACP) were treated as distinct metabolites, breaking the peptidoglycan biosynthesis chain required for biomass. Renaming restored the connection and growth went from 0.0 to 0.59 h⁻¹.

Koduru2022 was selected as the model to use for this project. The zero-growth defect was a namespace artefact and was easily resolved. The presence of 687 GPRs makes the model suitable for gene-level analysis and consistent with the modeling standards required for community simulation in MICOM.

## 5. Curation
Section 5 details the curation steps applied to the selected model.

### 5.1 Reload of the Selected Model
To ensure complete reproducibility, the curation process is initiated from the GEM original file, preventing the accumulation of changes on the model previously modified in previous sections.


In [7]:
# Reload clean — curation starts from the original file
model_lm = cobra.io.read_sbml_model('../models/raw/Koduru2022.xml')
sol_base = model_lm.optimize()
print(f"Base growth: {sol_base.objective_value:.4f} h⁻¹  [{sol_base.status}]")

http://purl.obolibrary.org/obo/GO_0016052 does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id


Base growth: 0.0000 h⁻¹  [optimal]


The curation being made involves several steps:
- fix of peptido1_LME_c mismatch ID;
- namespace harmonization (e) -> _e
- glucose rename glc-D_e -> glc__D_e
- Off-flavors originated from LOX pathway (hexanal, nonenal)
- Strecker exports

### 5.2 Peptidoglycan ID fix
The `peptido1-LME_c`/`peptido1_LME_c` mismatch identified is corrected here by merging both entries into a single metabolite, restoring the link between PPTGS and SDADACP.

In [8]:
# peptidoglycan ID fix — merge peptido1-LME_c into peptido1_LME_c
met_correct = model_lm.metabolites.get_by_id('peptido1_LME_c')   # consumed by SDADACP
met_wrong  = model_lm.metabolites.get_by_id('peptido1-LME_c')   # produced by PPTGS

# replaces the wrong metabolite with the correct one in all its reactions
for rxn in list(met_wrong.reactions):
    stoich = rxn.metabolites[met_wrong]
    rxn.subtract_metabolites({met_wrong: stoich})
    rxn.add_metabolites({met_correct: stoich})

# remove duplicate metabolite
model_lm.remove_metabolites([met_wrong])

# to confirm if the reaction has the correct metabolite ID
pptgs = model_lm.reactions.get_by_id('PPTGS')
print(f"PPTGS: {pptgs.reaction}")

sol = model_lm.optimize()
print(f"Growth after fixing mismatch ID: {sol.objective_value}")


PPTGS: uaaGgla_c --> peptido1_LME_c + udcpdp_c
Growth after fixing mismatch ID: 0.5588543531190318


### 5.3 Namespace Harmonization

Koduru2022 encodes extracellular compartments with the suffix `(e)` (e.g. `glc-D(e)`), while the rest of the curation pipeline and the *L.plantarum* model (iLP728) follow the BiGG convention `_e`. All extracellular metabolite and exchange reaction IDs are converted accordingly, with no change to stoichiometry or bounds. This step is a prerequisite for cross-model ID matching in SMETANA/MICOM.

In [9]:
# Namespace harmonization — (e) → _e for all metabolites and exchange reactions
for met in model_lm.metabolites:
    if met.id.endswith('(e)'):
        met.id = met.id[:-3].replace('-', '__') + '_e'

for rxn in model_lm.reactions:
    if rxn.id.startswith('EX_') and rxn.id.endswith('(e)'):
        rxn.id = rxn.id[:-3].replace('-', '__') + '_e'

sol = model_lm.optimize()
print(f"Growth after harmonization: {sol.objective_value:.4f} h⁻¹")


Growth after harmonization: 0.5589 h⁻¹


### 5.4 Glucose Rename

In [10]:
# Glucose rename — glc-D_e → glc__D_e (BiGG standard)
# Required for shared glucose pool in MICOM community model
old_met_id = 'glc-D_e'
new_met_id = 'glc__D_e'

met = model_lm.metabolites.get_by_id(old_met_id)
affected_reactions = [rxn.id for rxn in met.reactions]
print(f"Renaming metabolite {old_met_id!r} → {new_met_id!r}")
print(f"Affected reactions: {affected_reactions}")
met.id = new_met_id

rxn = model_lm.reactions.get_by_id('EX_glc_e')
rxn.id = 'EX_glc__D_e'
print(f"Renamed exchange: 'EX_glc_e' → 'EX_glc__D_e'")

sol = model_lm.optimize()
print(f"\nGrowth after rename: {sol.objective_value:.4f} h⁻¹")

assert set(affected_reactions) == {'GLCpts', 'GLCt2', 'TREHe', 'MALTe', 'EX_glc_e'}, \
    f"Unexpected affected reactions: {affected_reactions}"
print(f"✓ Exactly 5 reactions affected as expected")

Renaming metabolite 'glc-D_e' → 'glc__D_e'
Affected reactions: ['TREHe', 'GLCpts', 'EX_glc_e', 'MALTe', 'GLCt2']
Renamed exchange: 'EX_glc_e' → 'EX_glc__D_e'

Growth after rename: 0.5589 h⁻¹
✓ Exactly 5 reactions affected as expected


The namespace harmonization in the previous step converted all compartment suffixes from `(e)` to `_e`, but left one residual inconsistency: the glucose metabolite ID remains `glc-D_e` (hyphen) instead of the BiGG standard `glc__D_e` (double underscore). The exchange reaction is `EX_glc_e` instead of `EX_glc__D_e`.

This matters specifically for community modelling. When MICOM merges individual GEMs into a single LP, it identifies shared metabolites by
matching extracellular IDs across models. iLP728 uses `glc__D_e` and Koduru2022 uses `glc-D_e`. Without this correction, MICOM creates two
separate glucose pools — one per organism — so the two strains never compete for the same substrate. 

The fix affects exactly 5 reactions: GLCpts, GLCt2, TREHe, MALTe, and EX_glc_e. No stoichiometry is changed.

### 5.5 Off-flavor target audit
Before adding new biochemistry, the model is checked for which off-flavor related metabolites, transports, and exchanges already exist natively.

In [11]:
OFF_FLAVOR_TARGETS = {
    'hexanal':            {'met_c': 'hxa_c',   'met_e': 'hxa_e',   'ex': 'EX_hxa_e'},
    'hexanol':            {'met_c': 'hxol_c',  'met_e': 'hxol_e',  'ex': 'EX_hxol_e'},
    'nonenal':            {'met_c': 'nnl_c',   'met_e': 'nnl_e',   'ex': 'EX_nnl_e'},
    'nonenol':            {'met_c': 'nnol_c',  'met_e': 'nnol_e',  'ex': 'EX_nnol_e'},
    '3-methylbutanal':    {'met_c': '3mbal_c', 'met_e': '3mbal_e', 'ex': 'EX_3mbal_e'},
    '2-methylbutanal':    {'met_c': '2mbal_c', 'met_e': '2mbal_e', 'ex': 'EX_2mbal_e'},
    'isobutyraldehyde':   {'met_c': 'ibutld_c','met_e': 'ibutld_e','ex': 'EX_ibutld_e'},
    'phenylacetaldehyde': {'met_c': 'pacald_c','met_e': 'pacald_e','ex': 'EX_pacald_e'},
}

met_ids = {m.id for m in model_lm.metabolites}
rxn_ids = {r.id for r in model_lm.reactions}

print(f"Pre-curation audit — Koduru2022 ({len(model_lm.reactions)} rxns, {len(model_lm.metabolites)} mets)\n")
print(f"  {'Compound':<22} {'met_c':^8} {'met_e':^8} {'exchange':<16}  Status")
print(f"  {'─'*22} {'─'*8} {'─'*8} {'─'*16}  {'─'*25}")

for name, ids in OFF_FLAVOR_TARGETS.items():
    has_c  = ids['met_c'] in met_ids
    has_e  = ids['met_e'] in met_ids
    has_ex = ids['ex']    in rxn_ids
    c_mark  = '✓' if has_c  else '✗'
    e_mark  = '✓' if has_e  else '✗'
    if has_c and has_e and has_ex:
        status = 'present — no action needed'
    elif not has_c and not has_e and not has_ex:
        status = 'absent — will be added'
    elif has_c and not has_e:
        status = 'internal only — export needed'
    else:
        status = 'partial — review required'
    print(f"  {name:<22} {c_mark:^8} {e_mark:^8} {ids['ex']:<16}  {status}")

Pre-curation audit — Koduru2022 (1094 rxns, 914 mets)

  Compound                met_c    met_e   exchange          Status
  ────────────────────── ──────── ──────── ────────────────  ─────────────────────────
  hexanal                   ✗        ✗     EX_hxa_e          absent — will be added
  hexanol                   ✗        ✗     EX_hxol_e         absent — will be added
  nonenal                   ✗        ✗     EX_nnl_e          absent — will be added
  nonenol                   ✗        ✗     EX_nnol_e         absent — will be added
  3-methylbutanal           ✓        ✗     EX_3mbal_e        internal only — export needed
  2-methylbutanal           ✓        ✗     EX_2mbal_e        internal only — export needed
  isobutyraldehyde          ✓        ✗     EX_ibutld_e       internal only — export needed
  phenylacetaldehyde        ✓        ✗     EX_pacald_e       internal only — export needed


### 5.6 LOX pathway off-flavors - hexanal and nonenal

Hexanal and (E)-2-nonenal are lipid oxidation (LOX) products generated from linoleic and linolenic acid breakdown in legume matrices and are not native to Koduru2022 — section above confirmed their absence. 

Steps 1–4 below add the minimal biochemistry needed to track their fate during fermentation: intracellular and extracellular metabolite pools, passive diffusion transport (justified by their small, nonpolar structure, which crosses the membrane without a dedicated carrier), an NADH-dependent alcohol dehydrogenase (ADH) reducing each aldehyde to its corresponding alcohol, and exchange reactions kept closed by default. The ADH gene-protein-reaction rule (`LEUM_RS00695 or LEUM_RS09340`) reflects documented broad substrate specificity for linear aldehydes in *Leuconostoc*, rather than a hexanal- or nonenal-specific enzyme.

In [12]:
# Step 1 — LOX metabolites (intra and extracellular)
hxa_c  = cobra.Metabolite(id='hxa_c',  formula='C6H12O', name='Hexanal',        compartment='c', charge=0)
hxa_e  = cobra.Metabolite(id='hxa_e',  formula='C6H12O', name='Hexanal',        compartment='e', charge=0)
hxol_c = cobra.Metabolite(id='hxol_c', formula='C6H14O', name='1-Hexanol',      compartment='c', charge=0)
hxol_e = cobra.Metabolite(id='hxol_e', formula='C6H14O', name='1-Hexanol',      compartment='e', charge=0)
nnl_c  = cobra.Metabolite(id='nnl_c',  formula='C9H16O', name='(E)-2-Nonenal',  compartment='c', charge=0)
nnl_e  = cobra.Metabolite(id='nnl_e',  formula='C9H16O', name='(E)-2-Nonenal',  compartment='e', charge=0)
nnol_c = cobra.Metabolite(id='nnol_c', formula='C9H18O', name='(E)-2-Nonen-1-ol', compartment='c', charge=0)
nnol_e = cobra.Metabolite(id='nnol_e', formula='C9H18O', name='(E)-2-Nonen-1-ol', compartment='e', charge=0)

model_lm.add_metabolites([hxa_c, hxa_e, hxol_c, hxol_e, nnl_c, nnl_e, nnol_c, nnol_e])
print(f"LOX metabolites added. Total: {len(model_lm.metabolites)}")

LOX metabolites added. Total: 922


In [13]:
# Step 2 — Passive diffusion transports (nonpolar compounds, no active carrier)
hxat  = cobra.Reaction('HXAt');  hxat.name  = 'Hexanal transport (passive diffusion)';    hxat.bounds  = (-1000, 1000); hxat.add_metabolites({hxa_e: -1.0, hxa_c: 1.0})
hxolt = cobra.Reaction('HXOLt'); hxolt.name = '1-Hexanol transport (passive diffusion)';  hxolt.bounds = (-1000, 1000); hxolt.add_metabolites({hxol_c: -1.0, hxol_e: 1.0})
nnlt  = cobra.Reaction('NNLt');  nnlt.name  = '(E)-2-Nonenal transport (passive diffusion)'; nnlt.bounds = (-1000, 1000); nnlt.add_metabolites({nnl_e: -1.0, nnl_c: 1.0})
nnolt = cobra.Reaction('NNOLt'); nnolt.name = '(E)-2-Nonen-1-ol transport (passive diffusion)'; nnolt.bounds = (-1000, 1000); nnolt.add_metabolites({nnol_c: -1.0, nnol_e: 1.0})

model_lm.add_reactions([hxat, hxolt, nnlt, nnolt])
print("Transport reactions added.")

Transport reactions added.


In [14]:
# Step 3 — ADH reactions (aldehyde → alcohol, NADH-dependent)
# GPR: LEUM_RS00695 or LEUM_RS09340 — documented promiscuity for linear aldehydes
nadh_c = model_lm.metabolites.get_by_id('nadh_c')
nad_c  = model_lm.metabolites.get_by_id('nad_c')
h_c    = model_lm.metabolites.get_by_id('h_c')

hxaadh = cobra.Reaction('HXAADH')
hxaadh.name   = 'Hexanal reductase (ADH)'
hxaadh.bounds = (-1000, 1000)
hxaadh.add_metabolites({hxa_c: -1.0, nadh_c: -1.0, h_c: -1.0, hxol_c: 1.0, nad_c: 1.0})
hxaadh.gene_reaction_rule = 'LEUM_RS00695 or LEUM_RS09340'

nnladh = cobra.Reaction('NNLADH')
nnladh.name   = '(E)-2-Nonenal reductase (ADH)'
nnladh.bounds = (-1000, 1000)
nnladh.add_metabolites({nnl_c: -1.0, nadh_c: -1.0, h_c: -1.0, nnol_c: 1.0, nad_c: 1.0})
nnladh.gene_reaction_rule = 'LEUM_RS00695 or LEUM_RS09340'

model_lm.add_reactions([hxaadh, nnladh])
print(f"HXAADH: {hxaadh.reaction}")
print(f"NNLADH: {nnladh.reaction}")

HXAADH: h_c + hxa_c + nadh_c <=> hxol_c + nad_c
NNLADH: h_c + nadh_c + nnl_c <=> nad_c + nnol_c


In [15]:
# Step 4 — LOX exchanges (closed by default; opened by legume medium function)
ex_hxa  = cobra.Reaction('EX_hxa_e');  ex_hxa.name  = 'Hexanal exchange';       ex_hxa.bounds  = (0, 1000); ex_hxa.add_metabolites({hxa_e: -1.0})
ex_hxol = cobra.Reaction('EX_hxol_e'); ex_hxol.name = '1-Hexanol exchange';     ex_hxol.bounds = (0, 1000); ex_hxol.add_metabolites({hxol_e: -1.0})
ex_nnl  = cobra.Reaction('EX_nnl_e');  ex_nnl.name  = '(E)-2-Nonenal exchange'; ex_nnl.bounds  = (0, 1000); ex_nnl.add_metabolites({nnl_e: -1.0})
ex_nnol = cobra.Reaction('EX_nnol_e'); ex_nnol.name = '(E)-2-Nonen-1-ol exchange'; ex_nnol.bounds = (0, 1000); ex_nnol.add_metabolites({nnol_e: -1.0})

model_lm.add_reactions([ex_hxa, ex_hxol, ex_nnl, ex_nnol])
print("LOX exchanges added.")

LOX exchanges added.


### 5.7 Strecker Aldehydes - branched-chain and aromatic amino acid degradation

2-methylbutanal, 3-methylbutanal, isobutyraldehyde and phenylacetaldehyde are Strecker/Ehrlich pathway products of branched-chain (Leu, Ile, Val) and aromatic (Phe) amino acid catabolism. 

Unlike the LOX compounds above, these already exist as intracellular metabolites in Koduru2022 — Step 5 only verifies their presence, it does not introduce new biochemistry. Step 5b extends each one to the extracellular compartment with passive diffusion transport and a default-closed exchange.

In [16]:
# Step 5a: Strecker exports
# Strecker aldehydes already exist internally in Koduru2022.
# Only extracellular metabolite + transport + exchange are added.No new biochemistry.
interns_strecker = {
    '2mbal_c':  '2-methylbutanal',
    '3mbal_c':  '3-methylbutanal',
    'ibutld_c': 'Isobutyraldehyde',
    'pacald_c': 'Phenylacetaldehyde',
}

print("Verification — Strecker aldehydes present internally:")
for met_id, name in interns_strecker.items():
    try:
        met = model_lm.metabolites.get_by_id(met_id)
        print(f"  ✓ {met_id} ({name}) — reactions: {[r.id for r in met.reactions]}")
    except KeyError:
        print(f"  ✗ {met_id} NOT FOUND — check ID")

Verification — Strecker aldehydes present internally:
  ✓ 2mbal_c (2-methylbutanal) — reactions: ['3MOPDC', 'ALCD2MB']
  ✓ 3mbal_c (3-methylbutanal) — reactions: ['ALCD3MB', '4MOPDC']
  ✓ ibutld_c (Isobutyraldehyde) — reactions: ['3MOBDC', 'ALCDIB']
  ✓ pacald_c (Phenylacetaldehyde) — reactions: ['AALDH']


In [17]:
# Step 5b: Strecker extracellular + transports + exchanges
# Add extracellular metabolites, transports and exchanges for Strecker aldehydes

# Rename to match BiGG/Lp convention before extending to extracellular compartment
# 2mbal_c/3mbal_c lack the 'd' present in BiGG name_nclature (2mbald/3mbald),
# used in iLP728. Renaming now avoids creating separate, non-overlapping
# pools for the same metabolite when Lp and Lm are merged in SMETANA/MICOM.
rename_map = {'2mbal_c': '2mbald_c', '3mbal_c': '3mbald_c'}
for old_id, new_id in rename_map.items():
    met = model_lm.metabolites.get_by_id(old_id)
    met.id = new_id

met_2mbal_c  = model_lm.metabolites.get_by_id('2mbald_c')
met_3mbal_c  = model_lm.metabolites.get_by_id('3mbald_c')
met_ibutld_c = model_lm.metabolites.get_by_id('ibutld_c')
met_pacald_c = model_lm.metabolites.get_by_id('pacald_c')

met_2mbal_e  = cobra.Metabolite(id='2mbald_e',  formula=met_2mbal_c.formula,  name='2-Methylbutanal',   compartment='e', charge=0)
met_3mbal_e  = cobra.Metabolite(id='3mbald_e',  formula=met_3mbal_c.formula,  name='3-Methylbutanal',   compartment='e', charge=0)
met_ibutld_e = cobra.Metabolite(id='ibutld_e', formula=met_ibutld_c.formula, name='Isobutyraldehyde',  compartment='e', charge=0)
met_pacald_e = cobra.Metabolite(id='pacald_e', formula=met_pacald_c.formula, name='Phenylacetaldehyde',compartment='e', charge=0)

model_lm.add_metabolites([met_2mbal_e, met_3mbal_e, met_ibutld_e, met_pacald_e])

strecker = [
    ('2mbald', met_2mbal_c,  met_2mbal_e),
    ('3mbald', met_3mbal_c,  met_3mbal_e),
    ('ibutld', met_ibutld_c, met_ibutld_e),
    ('pacald', met_pacald_c, met_pacald_e),
]

novas_rxns = []
for name_, met_c, met_e in strecker:
    t  = cobra.Reaction(f'{name_.upper()}t')
    t.name   = f'{met_c.name} transport (passive diffusion)'
    t.bounds = (-1000, 1000)
    t.add_metabolites({met_c: -1.0, met_e: 1.0})

    ex = cobra.Reaction(f'EX_{name_}_e')
    ex.name   = f'{met_c.name} exchange'
    ex.bounds = (0, 1000)
    ex.add_metabolites({met_e: -1.0})

    novas_rxns.extend([t, ex])

model_lm.add_reactions(novas_rxns)
print(f"Strecker reactions added: {len(novas_rxns)}")
for rxn in novas_rxns:
    print(f"  {rxn.id}: {rxn.reaction}")

Strecker reactions added: 8
  2MBALDt: 2mbald_c <=> 2mbald_e
  EX_2mbald_e: 2mbald_e --> 
  3MBALDt: 3mbald_c <=> 3mbald_e
  EX_3mbald_e: 3mbald_e --> 
  IBUTLDt: ibutld_c <=> ibutld_e
  EX_ibutld_e: ibutld_e --> 
  PACALDt: pacald_c <=> pacald_e
  EX_pacald_e: pacald_e --> 


Two of the four Strecker aldehydes - `2mbal_c` and `3mbal_c` - use IDs that diverge from the BiGG convention applied in iLP728 (`2mbald_e`, `3mbald_e`, missing the trailing "d" for aldehyde). Following the same rationale applied to the glucose ID in Section 5.4, both metabolites are renamed to `2mbald_c`/`3mbald_c` before extracellular extension, ensuring a shared metabolite pool with Lp in the merged SMETANA/MICOM models rather than two organism-specific, non-interacting pools.

In [18]:
# TEST - check flux for ADH when there is hexanal uptake, and verify connectivity via FVA
with model_lm:
    model_lm.reactions.get_by_id('EX_hxa_e').lower_bound = -0.001  # test value

    sol_teste = model_lm.optimize()
    print(f"Growth with available hexanal: {sol_teste.objective_value:.4f} h⁻¹")

    if sol_teste.status == 'optimal':
        flux_adh  = sol_teste.fluxes.get('HXAADH', 0)
        flux_hxol = sol_teste.fluxes.get('EX_hxol_e', 0)
        flux_hxa  = sol_teste.fluxes.get('EX_hxa_e', 0)

        print(f"\nRelevant fluxes (growth-maximizing FBA):")
        print(f"  EX_hxa_e  (hexanal uptake):    {flux_hxa:.6f} mmol/gDW/h")
        print(f"  HXAADH    (ADH reduction):     {flux_adh:.6f} mmol/gDW/h")
        print(f"  EX_hxol_e (hexanol secretion): {flux_hxol:.6f} mmol/gDW/h")

        # FBA alone cannot distinguish "disconnected" from "connected but not optimal".
        # FVA on HXAADH checks whether nonzero flux is feasible at all, regardless of optimality.
        fva_adh = cobra.flux_analysis.flux_variability_analysis(
            model_lm, reaction_list=['HXAADH'], fraction_of_optimum=0
        )
        fmin, fmax = fva_adh.loc['HXAADH', 'minimum'], fva_adh.loc['HXAADH', 'maximum']
        print(f"\nFVA range for HXAADH: [{fmin:.6f}, {fmax:.6f}] mmol/gDW/h")

        if abs(flux_adh) > 1e-8:
            print("\nActive ADH reaction - hexanal is being reduced to hexanol under FBA.")
        elif abs(fmax - fmin) > 1e-8 or abs(fmax) > 1e-8:
            print("\nHXAADH is connected (nonzero flux feasible) but unused by growth-maximizing FBA.")
        else:
            print("\nHXAADH flux range is [0,0] - reaction is disconnected, check connectivity.")
    else:
        print(f"Status: {sol_teste.status}")

Growth with available hexanal: 0.5589 h⁻¹

Relevant fluxes (growth-maximizing FBA):
  EX_hxa_e  (hexanal uptake):    -0.001000 mmol/gDW/h
  HXAADH    (ADH reduction):     0.001000 mmol/gDW/h
  EX_hxol_e (hexanol secretion): 0.001000 mmol/gDW/h

FVA range for HXAADH: [0.000000, 0.001000] mmol/gDW/h

Active ADH reaction - hexanal is being reduced to hexanol under FBA.


In [19]:
# Check whether hexanal has any consumer besides HXAADH, to determine if the nonzero ADH flux above reflects a metabolic preference or a
# structural absence of alternative routing
hxa_c_met = model_lm.metabolites.get_by_id('hxa_c')
for rxn in hxa_c_met.reactions:
    print(f"{rxn.id}: {rxn.reaction}")

HXAt: hxa_e <=> hxa_c
HXAADH: h_c + hxa_c + nadh_c <=> hxol_c + nad_c


**Interpretation:**
Unlike the *L.plantarum* model, where FVA showed HXAADH connected but unused by growth-maximizing FBA, here FBA actively routes the imposed hexanal uptake through HXAADH to hexanol. 

This difference does not reflect a metabolic preference specific to *L. mesenteroides* — inspection of the metabolite network shows that HXAADH is the only reaction consuming `hxa_c` in this model; no alternative consuming reaction exists. With hexanal forced into the cytosol and no other sink available, the solver has no alternative routing, so nonzero ADH flux is a structural consequence of the curated network topology rather than evidence of an active or favourable LOX-derived aldehyde detoxification pathway. 

In [20]:
# Export final curated model — single output file, no intermediate versions
MODEL_OUT = '../models/curated/Koduru2022_curated_v2.xml'

cobra.io.write_sbml_model(model_lm, MODEL_OUT)

print(f"Exported: {MODEL_OUT}")
print(f"  Reactions:   {len(model_lm.reactions)}")
print(f"  Metabolites: {len(model_lm.metabolites)}")
print(f"  Genes:       {len(model_lm.genes)}")
fva = cobra.flux_analysis.flux_variability_analysis(model_lm, fraction_of_optimum=0)
blocked = fva[(fva.maximum == 0) & (fva.minimum == 0)]
print(f" {len(blocked)} / {len(model_lm.reactions)}  ({len(blocked)/len(model_lm.reactions)*100:.1f}%)")

sol_final = model_lm.optimize()
print(f"  Growth:      {sol_final.objective_value:.4f} h⁻¹  [{sol_final.status}]")

Exported: ../models/curated/Koduru2022_curated_v2.xml
  Reactions:   1112
  Metabolites: 926
  Genes:       687
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpzsk7q90w.lp
Reading time = 0.00 seconds
: 927 rows, 2225 columns, 9303 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpf9603rpa.lp
Reading time = 0.00 seconds
: 927 rows, 2225 columns, 9303 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in

## 6. Interpretation & Limitations

The curated Koduru2022 model (`Koduru2022_curated_v2.xml`) contains 1112 reactions and recovers growth of 0.5589 h⁻¹ on the default medium, consistent with the value obtained immediately after the peptidoglycan ID fix in Section 5.2 - confirming that subsequent curation steps (namespace harmonization, off-flavor pathway additions) did not alter core network function.

The ADH connectivity test showed FVA-feasible flux through HXAADH but, unlike the equivalent test in the *L.plantarum* model, FBA actively routed flux through it once hexanal uptake was forced. Inspection of the metabolite network showed this is a structural artefact, not a metabolic preference: HXAADH is the only reaction consuming `hxa_c` in this model, so any hexanal forced into the cytosol has no alternative fate. The test therefore isolates pathway connectivity under an artificial uptake bound; it does not establish that hexanal detoxification is an active or favoured process under nutrient-replete legume fermentation conditions.

Two Strecker aldehyde IDs (`2mbal_c`, `3mbal_c`) were renamed to the BiGG-consistent `2mbald_c`/`3mbald_c` (Section 5.7) to avoid creating organism-specific, non-overlapping metabolite pools when merged with iLP728 in SMETANA and MICOM.

## 7. Bibliography

1. Fernando Cruz, João Capela, Eugénio C. Ferreira, Miguel Rocha, Oscar Dias. BioISO: an objective-oriented application for assisting the curation of genome-scale metabolic models. bioRxiv 2021.03.07.434259; doi: https://doi.org/10.1101/2021.03.07.434259

2. Tao, A. et al. (2022). Mechanism and application of fermentation to remove beany flavor from plant-based meat analogs: A mini review. Frontiers in Microbiology, 13, 1070773. doi: 10.3389/fmicb.2022.1070773

3. Koduru, L., Kim, Y., Bang, J., Lakshmanan, M., Han, N. S., & Lee, D. Y. (2017). Genome-scale modeling and transcriptome analysis of Leuconostoc mesenteroides unravel the redox governed metabolic states in obligate heterofermentative lactic acid bacteria. Scientific reports, 7(1), 15721. https://doi.org/10.1038/s41598-017-16026-9

4. Koduru L, Lakshmanan M, Lee YQ, Ho PL, Lim PY, Ler WX, Ng SK, Kim D, Park DS, Banu M, Ow DSW, Lee DY. (2022). Systematic evaluation of genome-wide metabolic landscapes in lactic acid bacteria reveals diet- and strain-specific probiotic idiosyncrasies. Cell Rep. 2022 Dec 6;41(10):111735. doi: 10.1016/j.celrep.2022.111735. PMID: 36476869.